# Week 3 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-03.md`](../../weeks/week-03.md) · **Given engines:** [`modes.py`](modes.py), [`mdhash.py`](mdhash.py)

Teaching walkthrough of the week-3 studio. It **imports the reference exploits from
[`solution.py`](solution.py)** — never re-pasting them — so what runs here is exactly
what `test_modes.py` grades. Correctness is verified separately:
`python3 studios/_verify_solutions.py week-03`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution/modes/mdhash/data) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-03"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import os, inspect
import modes, mdhash
import data
import solution

# The theme: AES and SHA-256 stay intact all week. Every break is a misuse of the
# MODE or the CONSTRUCTION wrapped around an unbroken primitive.

## Task 1 — ECB leaks structure; CBC hides it (same cipher, different mode)

A block cipher is deterministic: identical plaintext blocks map to identical
ciphertext blocks. So `ecb_leak_count` — distinct *ciphertext* blocks under ECB —
equals the distinct *plaintext* blocks; the structure leaks 1:1. The provided
`IMAGE` has two flat regions ⇒ 2 distinct blocks. CBC chaining over the *same*
image and cipher scrambles every block ⇒ 96. "We use AES" tells you nothing until
you know the mode.

In [ ]:
print(inspect.getsource(solution.ecb_leak_count))
# mirror test_ecb_leaks_structure_cbc_hides_it
key = os.urandom(16)
ecb_distinct = solution.ecb_leak_count(data.IMAGE, key)
cbc_distinct = modes.distinct_blocks(modes.cbc_encrypt(data.IMAGE, key, os.urandom(3)))
assert ecb_distinct == data.ECB_DISTINCT_EXPECTED == 2, ecb_distinct
assert cbc_distinct == data.CBC_DISTINCT_EXPECTED == 96, cbc_distinct
print(f"ECB leaks {ecb_distinct} distinct blocks; CBC hides structure ({cbc_distinct})")
print("the MODE decided, not the cipher")

## Task 2 — CTR nonce reuse *is* week 2's two-time pad

CTR turns a block cipher into a stream cipher: `c = m ⊕ keystream(key, nonce)`.
Reuse the nonce under one key and both messages share a keystream, which cancels:
`c1 ⊕ c2 == m1 ⊕ m2`, so `m2 = c1 ⊕ c2 ⊕ m1`. Verbatim week 2, on a modern mode.
The strong cipher gives *zero* protection — its confidentiality was conditional on
nonce uniqueness (a real, recurring CVE class).

In [ ]:
print(inspect.getsource(solution.recover_second_plaintext))
# mirror test_ctr_nonce_reuse_recovers_plaintext
key, nonce = os.urandom(16), os.urandom(8)   # nonce REUSED across both messages
ks = modes.ctr_keystream(key, nonce, max(len(data.M1), len(data.M2)))
c1, c2 = modes.xor(data.M1, ks), modes.xor(data.M2, ks)
assert modes.xor(c1, c2) == modes.xor(data.M1, data.M2)   # keystream cancels
recovered = solution.recover_second_plaintext(c1, c2, data.M1)
assert recovered == data.M2, recovered
print("recovered m2:", recovered.decode())

## Task 3 — forge a `H(secret‖msg)` MAC by length extension

A Merkle–Damgard digest *is* the full internal state, so hashing can RESUME from a
tag. Knowing only `(msg, tag, len(secret))` — never the secret — the attacker
replicates the hash's glue padding and continues hashing from the observed tag,
producing a valid tag for `msg ‖ pad ‖ extension`. `good_mac` (real HMAC) nests the
hashing, so the resumable inner state is never exposed: same attack, no path.

In [ ]:
print(inspect.getsource(solution.forge_extension))

### Watch the guarantee fail — live

This reproduces `test_length_extension_breaks_bad_mac_guarantee`. The
`H(secret‖msg)` construction *claims* only the key-holder can produce a valid tag.
The forge succeeds without the key — the guarantee collapses. HMAC rejects the
same forgery. **Control Scorecard terms:** the primitive (SHA-256-like hash) was
never broken; the naive *construction* only ever RAISED COST and its authenticity
was not a real GUARANTEE. The fix is a better construction, not a better hash.

In [ ]:
tag = mdhash.bad_mac(data.MAC_SECRET, data.MAC_MSG)      # attacker sees (msg, tag)
forged_msg, forged_tag = solution.forge_extension(
    data.MAC_MSG, tag, len(data.MAC_SECRET), data.MAC_EXTENSION)
# the server, holding the secret, computes the SAME tag for forged_msg
server_tag = mdhash.bad_mac(data.MAC_SECRET, forged_msg)
assert forged_tag == server_tag, "forge must match the server's tag"
assert data.MAC_EXTENSION in forged_msg
print(f"forged msg: {forged_msg!r}")
print(f"forged tag {forged_tag} == server tag {server_tag}: VALID forgery, no secret")
print()

# HMAC defeats it: a keyless 'extension' does not validate.
import hmac
forged_guess = mdhash.good_mac(b"", data.MAC_MSG + data.MAC_EXTENSION)
real_tag     = mdhash.good_mac(data.MAC_SECRET, data.MAC_MSG + data.MAC_EXTENSION)
assert not hmac.compare_digest(forged_guess, real_tag)
print("HMAC rejected the same forgery")
print()
print("LESSON (scorecard axis 2): H(secret||msg) authenticity was RAISES-COST, not a")
print("GUARANTEE -- length extension forges a valid tag with no key. HMAC's")
print("authenticity IS a guarantee under a secret key. Better construction, not hash.")